# Eksperimentere med Neo4j
Sjekke at vi kan kjøre Neo4J

Starte med
```
sudo apt install podman
```
Her kjører vi på egne maskiner heller enn å benytte PITs test-server.  Årsaken er enkel: Fungerer uten VDI.

## Installere Neo4J

Her kommer koden for å starte Neo4J.  Kan kjøres her, men jeg liker å ha det i en egen terminal for å kunne manipulere utenfor *notebook*.

In [ ]:
%%bash
PWD=$(pwd)
mkdir -p neo4j
mkdir -p neo4j/data
mkdir -p neo4j/logspo
mkdir -p neo4j/plugins
mkdir -p neo4j/import

# Dette laster ned to plugins fra Neo4j (som må importeres for hånd om vi skal kjøre på VDI)

podman run \
    -p 7474:7474 -p 7687:7687 \
    --userns=keep-id \
    -e NEO4J_PLUGINS='["apoc", "graph-data-science"]' \
    -e NEO4J_dbms_security_procedures_unrestricted='gds.*,apoc.*' \
    -e NEO4J_dbms_security_procedures_allowlist='gds.*,apoc.*' \
    -e NEO4J_apoc_import_file_enabled=true \
    -v $PWD/neo4j/data:/data:Z \
    -v $PWD/neo4j/logs:/logs:Z \
    -v $PWD/neo4j/import:/import:Z \
    -v $PWD/neo4j/plugins:/plugins:Z \
    -e NEO4J_AUTH=neo4j/password \
    -d docker.io/library/neo4j:latest

: 

Når man er ferdig er det bare å kopiere identifikatoren over inn i neste kall

In [163]:
%%bash
podman kill d1349c6d65b5e1ac85ac00826fbe43f4e4e51a6553dd897b456c304a2d69734d

d1349c6d65b5e1ac85ac00826fbe43f4e4e51a6553dd897b456c304a2d69734d


Gi databasen litt tid til å starte

Grafen er nå tilgjengelig på 
```
http://localhost:7474/browser/
```


Vi skal stort sett programmere mot Neo4j og bare unntaksvis bruke nettleseren til å "se" på grafer.

Kontakt!


## (Kort) introduksjon til Cypher

Cypher er "SQL for grafer".  Det er strukturert på samme måte, ved at data (muligens etter transformasjoner) "flyter" gjennom programmet.  Neo4j har støtte for alt man ønsker seg, så som transaksjoner, men vi skal bare skrape tilstrekkelig på overflaten til å kunne arbeide videre på egen hånd.


### Om noder

Noder i grafen kan ha (det vi kan kalle) "typer" eller merkelapper (*labels*).  En node av type person skrives slik `(:Person)`; det er parantesen som forteller at dette er en node.  Noder kan i tillegg ha egenskaper (*attributes*): `(:Person {navn:"TaSK", alder:42})`.

### Om relasjoner

Noder har relasjoner til hverandre.  På samme måte som noder kan kanter type og egenskaper.  `[:Ansatt {begynte: 2019, stilling:"TechLead"}]` er en kant (mellom to noder).

Kanter har retning, men når man søker er det ikke nødvendig å angi retningen (man får treff begge veier).  Eksempler nedenfor

### Cypher

#### Noder
Åpne nettleseren på adressen `http://localhost:7474/browser/`.  Dette er den interaktive måten å arbeide med en graf.

La oss lage en node; skriv:
```
    CREATE (n:Person {navn: "TaSK", alder: 42}) return n
```
Du får en node.  Klikk på den og se informasjonen du la inn.  Legg merke til at ute til venstre kan du velge mellom å "se" på noden, eller å få det som data, eller som "tabell".  Vi skal se nærmere på dette når vi kaller på databasen.

Lag en ny node av en annen type:
```
    CREATE (n:Firma {navn: "PIT", sektor: "Offentlig"}) return n
```
La oss søke etter alle noder i databasen
```
    MATCH (n) return n
```
Søk etter en node av en valgt type:
```
    MATCH (n:Person) return n
```
Finne to noder av forskjellig type:
```
    MATCH (n1:Person), (n2:Firma) return n1, n2
```
Det reserverte ordet `MATCH` tilsvarer på mange måter `SELECT`.

#### Kanter (relasjoner)
La oss knytte de to nodene våre sammen:
```
    MATCH (n1:Person)
    MATCH (n2:Firma)
    MERGE (n1) - [r:Jobber {ansatt: 2019}] -> (n2)
    RETURN n1, n2, r
```
`MERGEP` oppretter relasjonen dersom den ikke finnes; alternativet er `CREATE`.
Klikk på "table" ute til venstre og se detaljene.

Om vi hadde hatt mange noder av hver type ville vi fått mange relasjoner; det gjelder å kunne skille noder fra hverandre!

### Sletting
Vi sletter relasjonene, og sletter nodene
```
    MATCH (n) DETACH DELETE n
```

### Gjøre det fra Python

Vi starter med å opprette kontakt med databasen.  Port 7687 er BOLT, en binær protokoll som biblioteket pakker ut for oss.

In [ ]:
# Opprette forbindelse til databasen
from neo4j import GraphDatabase

URI = "bolt://localhost:7687"
AUTH = ("neo4j", "password")

driver = GraphDatabase.driver(URI, auth=AUTH)
driver.verify_connectivity()
records, summary, keys = driver.execute_query(
    """RETURN apoc.version() AS version;""")
print(f"Server Address: {summary.server.address}")

Server Address: 127.0.0.1:7687
Kontakt!


Når vi kaller (biblioteket som kaller) databasen får vi tre elementer tilbake:
- `records` er det vi returnerer i Cypher.  For eksempel `MATCH (p:Person) RETURN p.navn AS navn, p.uid AS ID` så vil vi finne `navn` og `ID` i `records`;
- `summary` er meta-informasjon om spørringen, slikt som `summary.counters.nodes_created`, og
- `keys` er rett og slett navnene vi har gitt returnverdiene.  Det vil si at om `RETURN p.navn, p.ID` så vil `keys` være `['p.navn', 'p.ID']`.  Tenk på disse som navn på kolonnene dersom vi ser på `records` som data (kolonner).

Opprette to noder og en relasjon mellom dem:

In [ ]:
records, summary, keys = driver.execute_query(
    """
    CREATE 
        (n1:Person {navn: 'TaSK', alder: 42})
        -[r:jobber {ansatt: 2019}]->
        (n2:Firma {navn: 'PIT', sektor: 'Offentlig'})
        RETURN n1 as person, n2 as jobb, r as relasjon
    """)
#
for record in records:
    record_dict = record.data()
#
for r in record_dict:
    print(f"\t{r}: {record_dict[r]}")
#
print("Keys:")
for k in keys:
    print(f"\t{k}")
#
print("Grafen")
print(f"\tNye noder: {summary.counters.nodes_created}")
print(f"\tNye kanter: {summary.counters.relationships_created}")
      
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")

	person: {'navn': 'TaSK', 'alder': 42}
	jobb: {'sektor': 'Offentlig', 'navn': 'PIT'}
	relasjon: ({'navn': 'TaSK', 'alder': 42}, 'jobber', {'sektor': 'Offentlig', 'navn': 'PIT'})
Keys:
	person
	jobb
	relasjon
Grafen
	Nye noder: 2
	Nye kanter: 1
Ressursbruk
	Kjøringen: 41ms
	Å konsumere: 1ms


In [8]:
# Søke etter en node
records, summary, keys = driver.execute_query(
    """
    MATCH (n1:Person) -[r:jobber]-> (n2)
    WHERE r.ansatt > 2010
    AND n2.sektor = "Offentlig"
    RETURN n2
    """)
#
for record in records:
    record_dict = record.data()
#
for r in record_dict:
    print(f"\t{r}: {record_dict[r]}")
#
print("Keys:")
for k in keys:
    print(f"\t{k}")
#
print("Grafen")
print(f"\tNye noder: {summary.counters.nodes_created}")
print(f"\tNye kanter: {summary.counters.relationships_created}")
      
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")

	n2: {'sektor': 'Offentlig', 'navn': 'PIT'}
Keys:
	n2
Grafen
	Nye noder: 0
	Nye kanter: 0
Ressursbruk
	Kjøringen: 86ms
	Å konsumere: 9ms


Denne siste kan man like godt kjøre i nettleseren.  Daw får man én node.  Ved å klikke på "grafen" under noden får man de "omkringliggende" nodene (som i dette tilfelle er kun én).

In [ ]:
# Slette det vi har laget og være klar til neste steg
records, summary, keys  = driver.execute_query(
    """
    MATCH (n) DETACH DELETE n
    RETURN COUNT(n)
    """)
#
for record in records:
    record_dict = record.data()
#
print("Resultat:")
for r in record_dict:
    print(f"\t{r}: {record_dict[r]}")
#
print("Keys:")
for k in keys:
    print(f"\t{k}")
#

Resultat:
	COUNT(n): 0
Keys:
	COUNT(n)


## APOC

Som SQL er Cypher rettet mot operasjonen på objekter i en database,  Men livet er så mye mer.  *Awesome Procedures on Cypher* er et stort bibliotek med hundrevis av rutiner for å gjøre "alt det andre".  Det ble lastet ned da vi startet databasen.  La oss sjekke at det hos oss.

In [15]:
records, summary, keys = driver.execute_query(
    """RETURN apoc.version() AS version;""")
print("Keys:")
for k in range(len(keys)):
    print(f"\t{keys[k]}: {records[k]}")
#

Keys:
	version: <Record version='2025.11.2'>


In [16]:
# Være sikker på at vi ikke starter med gamle data
records, summary, keys = driver.execute_query(
    """match (n) detach delete n""")
#
for s in summary.gql_status_objects:
    print(s)
#
print(f"Antall noder slettet: {summary.counters.nodes_deleted}")
print(f"Antall kanter slettet: {summary.counters.relationships_deleted}")

note: successful completion - omitted result
Antall noder slettet: 0
Antall kanter slettet: 0


## Fra Networkx til Neo4j

Da skal vi bruke APOC til å lese inn en stor graf, og se litt på den.

### Eksportere fra Networkx

In [148]:
import gzip
import networkx as nx

with gzip.open("data/email.edgelist.txt.gz", "rt") as fd:
    G = nx.read_edgelist(fd, create_using=nx.DiGraph())
#
G.remove_edges_from(nx.selfloop_edges(G))
print(f"Noder: {G.number_of_nodes()}")
print(f"Kanter: {G.number_of_edges()}")

nx.set_node_attributes(G, ":Person", name="labels")
nx.set_edge_attributes(G, "EPOST", name="label")

for n, d in G.nodes(data=True):
    # n er str
    G.nodes[n]["Navn"] = "Bruker " + n
#

# Skriv ut grafen
nx.write_graphml(G, "neo4j/import/large_graph.graphml", named_key_ids=True)
print("ok")

Nodes: 57194
Edges: 103083
ok


### Lese inn i Neo4j

In [149]:
records, summary, keys = driver.execute_query(
    """CALL apoc.import.graphml("large_graph.graphml", {storeNodeIds: true, readLabels: true})""")
# for enkelt å pakke opp svaret
for record in records:
    record_dict = record.data()
#
for k in record_dict:
    print(f"\t{k}: {record_dict[k]}")
#
print("Grafen")
print(f"\tNye noder: {summary.counters.nodes_created}")
print(f"\tNye kanter: {summary.counters.relationships_created}")
print("Begge er 0 fordi dette bare gir mening når Cypher koden Per Se genererer noder")
      
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")

	file: large_graph.graphml
	source: file
	format: graphml
	nodes: 57194
	relationships: 103083
	properties: 57194
	time: -248
	rows: 0
	batchSize: -1
	batches: 0
	done: True
	data: None
Grafen
	Nye noder: 0
	Nye kanter: 0
Begge er 0 fordi dette bare gir mening når Cypher koden Per Se genererer noder
Ressursbruk
	Kjøringen: 83ms
	Å konsumere: -241ms


## Ting og tang

### Sette en index

In [ ]:
records, summary, keys = driver.execute_query(
    """
    CREATE INDEX
    IF NOT EXISTS 
    FOR (n:Person) ON (n.Navn);
    """)      
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")

Ressursbruk
	Kjøringen: 12ms
ok


### Hente data
Sjekke noen velkjente noder:
- Bruker 23
- Bruker 26
- Bruker 15
- Bruker 6

Den "andre" måten å bruke driveren på, er gjennom sesjoner.

Resultatet må hentes i konteksten

In [152]:
with driver.session(database="neo4j") as session:
    resultat = session.run("""
    MATCH (p:Person)
    WHERE p.Navn IN ["Bruker 23", "Bruker 26", "Bruker 15", "Bruker 6"]
    RETURN p.Navn AS navn, COUNT {(p)--()} as naboer
    """)
    det_hele = []
    for r in resultat:
        det_hele += [r]
    #
    for s in resultat.consume().gql_status_objects:
        print(f"Hvordan gikk det: {s}")
    #
#
for d in det_hele:
    print(d)


Hvordan gikk det: note: successful completion
<Record navn='Bruker 23' naboer=137>
<Record navn='Bruker 26' naboer=99>
<Record navn='Bruker 15' naboer=75>
<Record navn='Bruker 6' naboer=263>


La oss ha antall kanter lett tilgjengelig

In [153]:
# Legg inn
records, summary, keys = driver.execute_query(
    """
    MATCH (n:Person)
    SET n.antallKanter = COUNT { (n)--() }
    """)
for r in records:
    # Skal være tom
    innhold = r.data()
    print(innhold)
#
records, summary, keys = driver.execute_query(
    """
    CREATE INDEX
    IF NOT EXISTS 
    FOR (n:Person) ON (n.antallKanter);
    """)
for r in records:
    # Skal være tom
    innhold = r.data()
    print(innhold)
#
print("ok")

ok


In [154]:
records, summary, keys = driver.execute_query(
    """
    MATCH 
    p=()-->(:Person {Navn:"Bruker 6"}) 
    RETURN p;
""")
# Returnerer et sett av STIER som ender i noden identifisert med id=6
for r in records:
    innhold = r.data()
    print(innhold)
    break # Første er tilstrekkelig
#
records, summary, keys = driver.execute_query(
    """MATCH (p)-->(:Person {Navn:"Bruker 6"}) RETURN p;
""")
# Returnerer et sett av NODER som har relasjoner til noden identifisert med id=6
for r in records:
    innhold = r.data()
    print(innhold)
    break # Første er nok
#
    

{'p': [{'antallKanter': 1, 'Navn': 'Bruker 24805'}, 'EPOST', {'antallKanter': 263, 'Navn': 'Bruker 6'}]}
{'p': {'antallKanter': 1, 'Navn': 'Bruker 24805'}}


## GDS

Cypher egner seg til "enkle ting".  Det vil si transaksjoner på og med noder, men er ikke egnet til å implementere algoritmer.  GDS er imkplementert i Java og kjører inne i Neo4J og har fri tilgang til alle interne datastrukturer.

In [13]:
# Sjekke at GDS har blitt lastet ned riktig
records, summary, keys = driver.execute_query(
    """
    CALL gds.version();
    """)

print("Keys:")
for k in range(len(keys)):
    print(f"\t{keys[k]}: {records[k]}")
#

Keys:
	gdsVersion: <Record gdsVersion='2.24.0'>


In [ ]:
# For ordens skyld, i tilfelle databasen har blitt brukt, la oss slette
# alle GDS-grafer.  
# Detaljer om litt
records, summary, keys = driver.execute_query(
    """
    CALL gds.graph.list() YIELD graphName
    WITH graphName
    CALL gds.graph.drop(graphName) YIELD graphName AS borte
    RETURN borte
    """)
# Trolig tom
for r in records:
    innhold = r.data()
    print(innhold)
#
print("ok")


ok



Fordi mange graf-algoritmer i praksis bruker alle noder i grafen, er valget med å kjøre i en definert sub-graf og kreve at den er i hukommelsen, et design som gir mening.  Tross alt er tilgang til data i hukommelsen minst fire størrelsesordner raskere, og ett eneste søk på disk kan ødelegge alt.  For å gjøre dette mulig er flyten delt i tre, og eksplisitt funksjonalitet for testing tilgjengelig.

Flytens tre (fire) steg er:
- Konstruere (sub)grafen i hukommelsen ved å velge noder og relasjoner som er relevante.  Det gjøres enten med primitiver tilgjengelig i GDS dersom subgrafen skal bestå av  "enkle ting".  Eller brukes Cyhper til å velge ut noder og relasjoner;
- For sikkerhets skyld bør man be om et estimat på algoritmen som skal kjøres.  Estimatet gjøres ved å estimere algoritmen opp mot subgrafen som er laget;
- Kjøre algoritmen, og
- Fjerne subgrafen når det ikke lenger er brhov for den.

Algoritmen bør ha noen sideeffekter.  Det er fire måter å skape dem:
- **stream**: Data returneres til kalleren, som enten er Python-koden eller i nettleseren;
- **stats**: Returnerer statestikk (metainformasjon) heller enn noder og relasjoner;
- **mutate**: Skriver endringer tilbake til subgrafen i hukommelsen, og
- **write**: Skriver endringer tilbake i selve databasen.




### Eksempel på lasting med merkede noder

Det enkleste er å merke det man vil ha med, og så laste dem inn direkte.  Vi har allerede satt merkelappen `antallKanter` på hver node, (og satt på en index) så vi laster med dem.



#### Merke nodene

In [189]:
records, summary, keys = driver.execute_query(
    """
    MATCH (p:Person)
    WHERE p.antallKanter > 100
    SET p:Viktig;
    """)
for s in resultat.consume().gql_status_objects:
    print(f"Hvordan gikk det: {s}")
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print("ok") 

Hvordan gikk det: note: successful completion
Ressursbruk
	Kjøringen: 21ms
ok


#### Lage subgrafen
Så laster vi de nodene sammen med (kun) `EPOST`-relasjonene for det kan jo være andre relasjoner i databasen:

In [181]:
records, summary, keys = driver.execute_query(
    """
    CALL gds.graph.project(
        'ViktigGraf',
        ['Viktig'],    // Første "søk": Noder
        ['EPOST']      // Andre  "søk": Kanter
    )
    """)
for s in resultat.consume().gql_status_objects:
    print(f"Hvordan gikk det: {s}")
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print("ok") 


Hvordan gikk det: note: successful completion
Ressursbruk
	Kjøringen: 0ms
ok


In [ ]:
# Hva er status
records, summary, keys = driver.execute_query(
    """CALL 
        gds.graph.list('cliqueFinderGraph') 
        YIELD graphName, nodeCount, relationshipCount, schema
        RETURN graphName, nodeCount, relationshipCount, schema.nodes AS NodeLabels
    """
)
for r in records:
    innhold = r.data()
    print(innhold)
    break # Første er nok
#
print("ok") 

#### Et estimat på kjøringen

In [165]:
# Et estimat
records, summary, keys = driver.execute_query(
    """
    CALL gds.pageRank.stream.estimate (  // Få estimat på å sende resultatene tilbake til meg (stream)
        'ViktigGraf',
        {
        maxIterations: 20,   // Parametre for algoritmen
        dampingFactor: 0.85
        }
        )
    YIELD nodeCount, bytesMin, bytesMax, requiredMemory
    """)
for s in resultat.consume().gql_status_objects:
    print(f"Hvordan gikk det: {s}")
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
for r in records:
    innhold = r.data()
    print(innhold)
    break # Første er nok
#
print("ok") 


Hvordan gikk det: note: successful completion
Ressursbruk
	Kjøringen: 1ms
{'nodeCount': 209, 'bytesMin': 5872, 'bytesMax': 5872, 'requiredMemory': '5872 Bytes'}
ok


#### Hente data fra subgrafen

In [168]:
# La oss få data
records, summary, keys = driver.execute_query(
    """
    CALL gds.pageRank.stream('ViktigGraf')
    YIELD nodeId, score
    RETURN gds.util.asNode(nodeId).Navn AS navn, score
    ORDER BY score DESC
    LIMIT 5
    """)
for s in resultat.consume().gql_status_objects:
    print(f"Hvordan gikk det: {s}")
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
for r in records:
    innhold = r.data()
    print(innhold)
#
print("ok") 


Hvordan gikk det: note: successful completion
Ressursbruk
	Kjøringen: 0ms
{'navn': 'Bruker 11798', 'score': 8.272769620366098}
{'navn': 'Bruker 12586', 'score': 2.593843757509939}
{'navn': 'Bruker 14603', 'score': 2.475295656143867}
{'navn': 'Bruker 880', 'score': 1.6377652947797572}
{'navn': 'Bruker 737', 'score': 1.6068969822908044}
ok


_Score_ er større enn 0 fordi summen av alle blir $N$ (opprinnelig satt til 1 på alle noder).

#### Fjerne subgrafen

In [184]:
# Fjerne subgrafen
records, summary, keys = driver.execute_query(
    """
    CALL gds.graph.drop('ViktigGraf', false)  // false = Ikke få feil om den ikke finnes
    YIELD graphName
    """)
for s in resultat.consume().gql_status_objects:
    print(f"Hvordan gikk det: {s}")
#
if not records:
    print("Allerede slettet")
else:
    print(f"Navnet på subgrafen: {records}")
#
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")

print("ok") 



Hvordan gikk det: note: successful completion
Allerede slettet
Ressursbruk
	Kjøringen: 1ms
ok


#### Fjerne merkelappene

In [190]:
# Til slutt,gjerne merkelappen

records, summary, keys = driver.execute_query(
    """
    MATCH (p:Viktig) 
    REMOVE p:Viktig
    RETURN COUNT (p);
    """)
for s in resultat.consume().gql_status_objects:
    print(f"Hvordan gikk det: {s}")
#
for rec in records:
    for r in rec:
        print(f"Slettet: {r}")
#
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")

print("ok") 

Hvordan gikk det: note: successful completion
Slettet: 209
Ressursbruk
	Kjøringen: 28ms
ok



### Eksempel uten merkelapp

Vi kan kjøre Cypher som en del av byggingen av en sub-graf; dette er et KI-forslag.  Vi trenger to (2) Cypher-kall: Ett for noder og ett for kanter.
```
CALL gds.graph.project(
   // Her finner vi hvilke noder som skal være med
  'ViktigGraf',            // Navnet på subgrafen
  'MATCH (p:Person) 
   WHERE COUNT { (p)-[:EMAIL]-() } > 100   // Bare noder med mer enn 100 eposter
   RETURN elementId(p) AS id',             // sendes til neste steg
   // Her finner vi hvilke kanter som skal være med.
   // Denne krever to tellinger for å sikre at det ikke forsøkes å lage en relasjon til noder
   // som ikke er med (har mindre enn 100 eposter).
  'MATCH (p1:Person)-[r:EMAIL]-(p2:Person) 
   WHERE COUNT { (p1)-[:EMAIL]-() } > 100  // 
     AND COUNT { (p2)-[:EMAIL]-() } > 100
   RETURN elementId(p1) AS source, elementId(p2) AS target, type(r) AS type' // Relationship query
)
```

### Eksempel på å skrive tilbake til databsen

Her ser vi hvordan data skrives tilbake til de nodene det gjelder

```
CALL gds.pageRank.write(
  'ViktigGraf',
  {
    writeProperty: 'HvorViktig' // Hva skal egenskapen på noden hete
  }
)
YIELD nodePropertiesWritten, ranIterations // Dette er returverdien og ikke sideeffekten!
```
Så kan vi hente resultatene med Cypher:
```
MATCH (p:Person)
WHERE p.HvorViktig IS NOT NULL
RETURN p.Navn, p.HvorViktig
ORDER BY p.HvorViktig DESC
LIMIT 10
```


### GDS til å lage noe som ligner på nettleseren
Et større og mer komplisert eksempel.
Dette er hvordan Neo4J tegner grafen i en *browser*:
![Grafen](data/Neo4j-graf.png)

Det er åpenbart en sub-graf.  Vi skal forsøke å gjenskape dette bildet.

1. Lage en subgraf bestående av alle personer (nå er det ikke noe annet i denne grafen);
2. Finne klikker or merke nodene med hvilken klikk de er med i;
3. Telle opp hvor mange medlemmer det er i hver klikk, og legge det inn i hver node, og
4. Fjerne merkingen på klikker som er "små".

Når vi (i nettleseren) ber om å få se klikkene kommer bare de store (og "støyen" filtreres ut).

#### Lag subgrafen

Hente ut Personer og kanter.  Vi er ikke interessert i retningen på kanten.

In [ ]:
records, summary, keys = driver.execute_query(
    """
    // Subgrafen heter EpostNettverk
    CALL gds.graph.project(
        'EpostNettverk', // Navnet på subgrafen
        'Person',  // Første søk: Nodene
        {
            EPOST: {
                orientation: 'UNDIRECTED' // Andre søk: relasjonene.  Konverteres til uten retning
            }
        }
    )
    YIELD graphName, nodeCount, relationshipCount;
    """)
for s in resultat.consume().gql_status_objects:
    print(f"Hvordan gikk det: {s}")
#
# for enkelt å pakke opp svaret
for record in records:
    record_dict = record.data()
#
for k in record_dict:
    print(f"\t{k}: {record_dict[k]}")
#

print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")

print("ok") 

Hvordan gikk det: note: successful completion
	graphName: EpostNettverk
	nodeCount: 57194
	relationshipCount: 206166
Ressursbruk
	Kjøringen: 0ms
ok


#### Identifisere gjengene

In [ ]:
# Burk Louvain for å finne gjenger og skrtive dem tilbake i databasen
records, summary, keys = driver.execute_query(
    """
    CALL gds.louvain.write(
        'EpostNettverk', 
        {
            writeProperty: 'GjengID'  // For hver node,skrive hvilken gjeng noden er med i
        }
    )
    YIELD communityCount, modularity;
    """)
for s in resultat.consume().gql_status_objects:
    print(f"Hvordan gikk det: {s}")
#
# for enkelt å pakke opp svaret
for record in records:
    record_dict = record.data()
#
for k in record_dict:
    print(f"\t{k}: {record_dict[k]}")
#

print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")

print("ok") 


Hvordan gikk det: note: successful completion
	communityCount: 288
	modularity: 0.7458673476585517
Ressursbruk
	Kjøringen: 4ms
ok


#### Fjerne små gjenger

In [ ]:
# Fjerne gjenger med mindre enn 10 medlemmer (tilfeldig valgt tall)
records, summary, keys = driver.execute_query(
    """
    MATCH (n)
    WITH 
        n.GjengID AS antall,   // Samler sammen alle noder med samme GjengID
        count(n) AS medlemmer  // Teller hvor mange i hver gjeng
    WHERE medlemmer < 10
    MATCH (m {GjengID: antall})
    REMOVE m.GjengID;
    """)
for s in resultat.consume().gql_status_objects:
    print(f"Hvordan gikk det: {s}")
#
# for enkelt å pakke opp svaret
for record in records:
    record_dict = record.data()
#
for k in record_dict:
    print(f"\t{k}: {record_dict[k]}")
#

print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")

print("ok") 


Hvordan gikk det: note: successful completion
	communityCount: 288
	modularity: 0.7458673476585517
Ressursbruk
	Kjøringen: 1580ms
ok


Nå er det bare å limem dette inn i nettleseren for å få nesten samme bilde som over
```
MATCH (n) WHERE n.GjengID IS NOT NULL RETURN n LIMIT 1000
```


#### Fjerne subgrafen

In [204]:
# Fjerne subgrafen
records, summary, keys = driver.execute_query(
    """
    CALL gds.graph.drop('EpostNettverk', false)  // false = Ikke få feil om den ikke finnes
    YIELD graphName
    """)
for s in resultat.consume().gql_status_objects:
    print(f"Hvordan gikk det: {s}")
#
if not records:
    print("Allerede slettet")
else:
    print(f"Navnet på subgrafen: {records}")
#
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")

print("ok") 



Hvordan gikk det: note: successful completion
Allerede slettet
Ressursbruk
	Kjøringen: 0ms
ok
